In [ ]:
# Purpose - assess RH conditions at different y cuts (with correct normalizations)

findex = 163 #143 #113 # 64  63 #120


In [ ]:
# Imports and Folder Selection

import sys; sys.path.append('..')

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

%matplotlib widget

plt.rcParams['text.usetex'] = True
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq
cc = run.cc
B0 = run.B0

#my_steps = run.steps[:15]
#my_steps = [run.steps[-1]]
my_steps = run.steps
my_steps = [run.steps[0]]
nsteps = len(my_steps)

print(run.steps)

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = True)

In [ ]:
# choices

option = 0

if (option == 0):
    descr = "minijet"
    xcut_y = 4475
    angle_in_deg = -77.04
elif (option == 1):
    descr = "plasmoid"
    xcut_y = 1752
    angle_in_deg = 0

# Shock Conditions

theta = angle_in_deg * 0.5 * np.pi / 180

nx, ny, nz = -np.cos(theta), -np.sin(theta), 0
tx, ty, tz = np.sin(theta), -np.cos(theta), 0 

plt.figure()
plt.title(boilerplate)
plt.quiver(0, 0, nx, ny, color='r', scale=1, scale_units='xy', angles='xy', label='n')
plt.quiver(0, 0, tx, ty, color='b', scale=1, scale_units='xy', angles='xy', label='t')
plt.xlim(-1, 1); plt.ylim(-1, 1)
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.gca().set_aspect('equal', adjustable='box')
plt.legend()
plt.show()

PTI = 5.0 / 3.0 # polytropic index

boilerplate = run.boilerplate + f"\n steps = {my_steps}, y = {xcut_y}, {descr}"

# n cross t is positive

In [ ]:
# Choose Regions of Interest

xlim = run.xlim
ylim = run.ylim
rect = [0, run.xlim, 0, run.ylim]

xstrips = [None] * nsteps

for ts in range(nsteps):

    roi = fq_cut(d.fields, ts, rect)

    #xcut_y = ylim / 4 * 0.75
    ycut_x = xlim / 2

    xstrips[ts] = fq_xcut(roi, xcut_y)

In [ ]:
strip = sum(xstrips) / len(xstrips)

plt.figure()
plt.plot(strip.x, strip.pb)
plt.title("Magnetic Pressure (pb)")
plt.xlabel("x")
plt.ylabel("pb")
plt.show()

crossing_index = np.where(np.diff(strip.pb > 0.5))[0][1]
crossing_x = strip.x[crossing_index]
print(f"pb first crosses 0.5 at x = {crossing_x.values}")

In [ ]:
# RH Conditions

X_CUTOFF = 0.95


def plot_squant(axis, quantity):
    right_val = strip[quantity].sel(x=X_CUTOFF * run.xlim, method="nearest")
    axis.plot(strip.x, strip[quantity] / right_val)


strip['v_n'] = strip.velx * nx + strip.vely * ny + strip.velz * nz # in units of c
strip['v_t'] = strip.velx * tx + strip.vely * ty + strip.velz * tz
strip['B_n'] = (strip.bx * nx + strip.by * ny + strip.bz * nz)
strip['B_t'] = (strip.bx * tx + strip.by * ty + strip.bz * tz)  

strip['pn'] = strip.px * nx ** 2 + strip.py * ny ** 2

strip['v_dot_B'] = strip.bx * strip.velx + strip.by * strip.vely + strip.bz * strip.velz
strip['v_sqrd'] = strip.velx ** 2 + strip.vely ** 2 + strip.velz ** 2

PNORM = 2 / run.sigma

fig, axes = plt.subplots(3, 2, figsize=(8, 4), sharex=True)
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

plt.subplots_adjust(hspace=0, wspace = 0.2)


eqn_fsize = 18

# Mass

strip['mass_flux'] = strip.dens * strip.v_n 

plot_squant(ax1, 'mass_flux')
#ax1.set_title(r"$\rho v_n$")
ax1.text(0.45, 0.10, r"$\rho v_n$", transform=ax1.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Pressure

ram = PNORM * strip.dens * strip.v_n ** 2
strip['pressure'] = ram + strip.pn + strip.pb #changed from px


# TODO fix px (think this has been fixed)

plot_squant(ax2, 'pressure')
#ax2.set_title(r"$\rho v_n^2 + P_g + P_B$")
ax2.set_ylim(0, 1.2)
ax2.text(0.35, 0.10, r"$\rho v_n^2 + P_g + P_B$", transform=ax2.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')


# Energy

strip['energy'] = strip.v_n * (0.5 * PNORM * strip.dens * strip.v_sqrd + \
                               PTI / (PTI - 1) * strip.pn + 2 * strip.pb) - \
                                2 * strip.B_n * strip.v_dot_B

plot_squant(ax3, 'energy')
#ax3.set_title("Energy")

energy_expression = r'$\rho v_n \left(\frac{v^2}{2} + \frac{\gamma}{\gamma - 1} \frac{P_g}{\rho} + \frac{B^2}{4 \pi \rho}\right) - \frac{B_n (\mathbf{v} \cdot \mathbf{B})}{4 \pi}$'
ax3.text(0.15, 0.10, energy_expression, transform=ax3.transAxes, fontsize=12, verticalalignment='bottom')

# B_N

plot_squant(ax4, 'B_n')
#ax4.set_title("$B_n$")
ax4.text(0.45, 0.10, '$B_n$', transform=ax4.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Poynting Flux

strip['poynting'] = PNORM * strip.dens * strip.v_n * strip.v_t - 2 * strip.B_n * strip.B_t
plot_squant(ax5, 'poynting')
#ax5.set_title(r"$\rho v_n v_t - B_n B_t / 4 \pi$")
ax5.text(0.35, 0.10, r'$\rho v_n v_t - \frac{B_n B_t} {4 \pi}$', transform=ax5.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

# Electric Field

strip['E_ideal'] = -strip.v_n * strip.B_t + strip.v_t * strip.B_n # I flipped this so it is Ez_ideal

plot_squant(ax6, 'E_ideal')
ax6.text(0.35, 0.10, '$B_n v_t - v_n B_t$', transform=ax6.transAxes, fontsize=eqn_fsize, verticalalignment='bottom')

#ax6.set_title("$B_n v_t - v_n B_t$")



for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(run.xlim / 2, run.xlim * X_CUTOFF)
        #ax.set_ylim(0, 2.4)
        boundary = strip.x[np.where(np.diff(strip.pb > 0.5))[0][1]]
        print(f"boundary is at {boundary}")
        ax.axvline(boundary, color = 'green', linestyle = '--')

  

for ax in [ax3, ax6]:
    ax.set_xlabel("$x$ [$c / \omega_p$]")

#plt.suptitle("Rankine-Hugoniot Conditions \n" + run.boilerplate + f"\n steps = {my_steps}, y = {xcut_y} ({xcut_y / run.ylim:.02f})")
#plt.suptitle("Rankine-Hugoniot Conditions")

plt.suptitle(boilerplate)


plt.show()

#save_polished_fig("rh")

print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])

In [ ]:
plt.rcParams.update({'font.size': 18})

#max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max() #NOTE different def
max_jz = strip.jz.max()

plt.figure()
#plt.axvline(boundary, color = 'green', linestyle = '--')
#plt.plot(strip.x, np.log(strip.dens), label = 'log n')
plt.plot(strip.x, strip.dens, label = 'n')
plt.plot(strip.x, -strip.B_t, label = 'B_t')


plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.title(boilerplate)
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
plt.rcParams.update({'font.size': 18})

#max_jz = strip.jz.sel(x = slice(len(strip.jz) / 2, len(strip.jz))).max() #NOTE different def
max_jz = strip.jz.max()

plt.figure()
plt.axvline(boundary, color = 'green', linestyle = '--')
plt.plot(strip.x, strip.dens, label = "$n / n_0$")
plt.plot(strip.x, strip.pn, label = "$P_g / P_{B0}$")
plt.plot(strip.x, strip.pb, label = "$P_B / P_{B0}$")
plt.plot(strip.x, strip.jz / max_jz, label = "$j_z / j_{\mathrm{max}}$") #/ max_jz
#plt.plot(strip.x, strip.jz * strip.ez / 1e-2, label = "$J_z E_z$") 


plt.xlim(run.xlim / 2, run.xlim * 0.95)
plt.xlabel("$x$ [$c / \omega_p$]")
plt.legend()
plt.show()
plt.tight_layout()

plt.rcParams.update({'font.size': 12})

#save_polished_fig("shock_cut")

In [ ]:
# Slow-Mode Shock Conditions

fig, axes = plt.subplots(3, 2, figsize=(8, 6))
ax1, ax2, ax3, ax4, ax5, ax6 = axes[0, 0], axes[1, 0], axes[2, 0], axes[0, 1], axes[1, 1], axes[2, 1]

def degree_formatter(x, pos):
    return f"{x:.0f}°"

strip['B_mag'] = strip.bx ** 2 + strip.by ** 2 + strip.bz ** 2
strip['theta_B'] = np.arccos(strip.B_n / strip.B_mag)

strip['v_A'] = np.sqrt(run.sigma) * strip.B_mag
strip['v_I'] = strip.v_A * np.cos(strip.theta_B)
strip['v_S'] = np.sqrt(run.sigma / 2 * PTI * strip.px / strip.dens)
strip['v_SM'] = np.sqrt(0.5) * np.sqrt(strip.v_S ** 2 + strip.v_A ** 2 - \
                                       np.sqrt((strip.v_S ** 2 + strip.v_A ** 2) ** 2 - 4 * strip.v_S ** 2 * strip.v_I ** 2))

strip['M_I'] = strip.v_n / strip.v_I 
strip['M_SM'] = strip.v_n / strip.v_SM

# Plasma Pressure

plot_squant(ax1, 'px')
ax1.set_title(r"$P_x$ (increase)")

# Magnetic Pressure

plot_squant(ax2, 'pb')
ax2.set_title(r"$P_B$ (decrease)")
ax2.set_ylim(0)

# Angle
strip['theta_B_deg'] = 180 / np.pi * np.arccos(strip.B_n / strip.B_mag)

plot_squant(ax3, 'theta_B_deg')
ax3.set_title(r"$\theta_B$ (decrease)")
ax3.yaxis.set_major_formatter(FuncFormatter(degree_formatter))

# M_I

plot_squant(ax4, 'M_I')
ax4.set_title(r"$M_I$ ($\leq 1$ upstream)")

# M_SM

plot_squant(ax5, 'M_SM')
ax5.set_title(r"$M_{SM}$ ($> 1 \; \rightarrow \; < 1$)")

for ax in [ax4, ax5]:
    ax.axhline(1, color = 'black', linestyle = '--')
    ax.axhline(0, color = 'grey', linestyle = ':', alpha = 1)

for ax_row in axes:
    for ax in ax_row:
        #ax.legend()
        ax.set_xlim(run.xlim / 2, run.xlim)

        for bound in strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)]:
            ax.axvline(bound, color = 'green', linestyle = '--')



plt.suptitle("Slow-Mode Shock Conditions \n" + run.boilerplate + f"\n steps = {my_steps}")

plt.tight_layout()
plt.show()



print(strip.x[np.isclose(strip.pb, 0.5, atol=1e-2)])